<h1 align='center'> ELCardio Competition </h1>

## Setup & Imports

In [ ]:
pip install iterative-stratification

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
import re
import nltk
import unicodedata
import itertools
import torch
import torch.nn.functional as F

from tqdm import tqdm
from torch import nn
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader, Dataset
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import precision_recall_fscore_support, classification_report, f1_score
from tqdm import trange
from nltk import tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import CountVectorizer
from dataclasses import dataclass
from sklearn.preprocessing import MultiLabelBinarizer
from typing import List, Dict, Optional, Iterator, Any

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Data Loading & Parsing

### Dataclasses

In [ ]:
@dataclass
class Mention:
    """
    Represents a mention-level annotation within the clinical text.
    
    Attributes:
        start (int): The starting character offset in the document.
        end (int): The ending character offset in the document.
        code (str): The assigned ICD-10 diagnostic code.
        text (str): The actual text span (e.g., "συγκοπτικό επεισόδιο").
    """
    start: int
    end: int
    code: str
    mention: str


@dataclass
class ClinicalDocument:
    """
    Represents a complete clinical document (discharge letter) from the dataset.
    
    Attributes:
        patient_id (Any): Unique identifier for the patient/record.
        text (str): The raw unstructured clinical narrative.
        document_labels (List[List[str]]): A list of lists containing valid ICD-10 codes 
            for the document. The system must predict at least one code from each inner list.
        mentions (Optional[List[Mention]]): A list of mention-level annotations, if available.
    """
    patient_id: Any
    text: str
    document_labels: List[List[str]]
    mentions: Optional[List[Mention]] = None

### Read from jsonl

In [ ]:
filepath = "/kaggle/input/datasets/dimitriospakakis/elcardio-training-set/trainELCardioCC2026.jsonl"
all_documents= []
with open(filepath, 'r', encoding='utf-8') as f:
    for line in f:
       json_line = json.loads(line)

       # Load the basic data
       patient_id = json_line['patient_id']
       text = json_line['text']
       document_level_annotations = json_line['document_level_annotations']

       # Load the mentions
       raw_mentions = json_line['mention_level_annotations'] 

       final_mentions = None
      
       # If the mentions list is not empty then we create Mention objects 
       if raw_mentions:
         final_mentions = [Mention(m['start'], m['end'], m['code'], m['mention']) for m in raw_mentions]
       
       all_documents.append(ClinicalDocument(patient_id,text,document_level_annotations,final_mentions))
        
print(len(all_documents))


In [ ]:
def fix_mention_offsets(document_text: str, mention: Mention) -> Mention:
    """
    Fix the wrong offsets looking for the rigth string in a window of +/- characters
    """
    # the correct string
    target_text = mention.mention
    
    # Define a search window
    search_start = max(0, mention.start - 15)
    search_end = min(len(document_text), mention.end + 15)
    search_window = document_text[search_start:search_end]
    
    # look up fot the string 
    local_start = search_window.find(target_text)
    
    if local_start != -1:
        # calculate the correct offsets
        new_start = search_start + local_start
        new_end = new_start + len(target_text)
        
        
        return Mention(start=new_start, end=new_end, code=mention.code, mention=target_text)
    else:
        return mention

In [ ]:
for document in all_documents:
    doc_mentions = document.mentions
    new_mentions = []
    if doc_mentions:
        for ment in doc_mentions:
          if not ment.end - ment.start < 2:
            new_mentions.append(fix_mention_offsets(document.text,ment))
    document.mentions=new_mentions


all_mentions = []
for doc in all_documents:
    if doc.mentions:
        all_mentions.append(doc.mentions)

print(f"{len(all_mentions)} mention-level annotations found")

In [ ]:
count= 0
for document in all_documents:
    ment = document.mentions
    if ment:
        for m in ment:
            new_m = fix_mention_offsets(document.text,m)
            st = new_m.start
            en = new_m.end
            txt = new_m.mention
            sliced_text = document.text[st:en]
            if sliced_text != txt:
                count+=1
print(f"{count} wrong offsets")

#### Remove overlaps

In [ ]:
overlaps_removed = 0

for doc in all_documents:
    if not doc.mentions:
        continue
        
    # Sort the mentions in a way that the longest mention is at the 1st position
    sorted_mentions = sorted(doc.mentions, key=lambda m: (m.start, -m.end))
    
    clean_mentions = []
    
    for m in sorted_mentions:
        if not clean_mentions:
            clean_mentions.append(m)
        else:
            last_mention = clean_mentions[-1]
            
            if m.start < last_mention.end:
                overlaps_removed += 1
                continue
            else:
                clean_mentions.append(m)
                
    doc.mentions = clean_mentions

print(f"{overlaps_removed} redundant/overlapping mentions were deleted.")

## Dataset Split

We will split the dataset to 90% train and 5% validation and 5% test. We will handle the codes that are rare. If a code has only 1 document, it will be in the training set. If there are more than 2 documents then there will be an uniform distribution in both test, train and validation sets

We can not convert the y_labels list to numpy array or tensor because every document has a different number of codes, but the 2nd dimension must be the same for all the samples. So, to improve the performance of the training and to solve this issue, we will convert the codes with multi-Hot Encoding. Each label list will be converted to a list where the true codes will be converted to 1 and the false to 0.

In [ ]:
X_texts = []
y_labels_raw = []
y_labels = []
for doc in all_documents:
    X_texts.append(doc.text)
    
    current_labels = []
    for label_list in doc.document_labels:
        for label in label_list:
            current_labels.append(label)
    y_labels.append(current_labels.copy())
    # Add extra flag to stratify the set using the labels and the mentions
    if doc.mentions and len(doc.mentions) > 0:
        current_labels.append("HAS_MENTION")
        
    y_labels_raw.append(current_labels)

X_texts = np.array(X_texts)

# Binarization
mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(y_labels_raw)

mlb2 = MultiLabelBinarizer()
y_labels = mlb2.fit_transform(y_labels)

# 3. Stratified Split (90% Train, 5% Val, 5% Test)
X_indices = np.arange(len(all_documents))

# --- Split: 90% Train / 10% Temp ---
msss_1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, temp_idx = next(msss_1.split(X_indices, y_encoded))

# --- Split: 5% Val / 5% Test ---
msss_2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_local_idx, test_local_idx = next(msss_2.split(temp_idx, y_encoded[temp_idx]))

val_idx = temp_idx[val_local_idx]
test_idx = temp_idx[test_local_idx]

train_docs = [all_documents[i] for i in train_idx]
val_docs = [all_documents[i] for i in val_idx]
test_docs = [all_documents[i] for i in test_idx]

y_train = y_labels[train_idx]
y_val = y_labels[val_idx]
y_test = y_labels[test_idx]

string_labels = mlb2.classes_
# Εκτυπώσεις
print(f"y_labels shape: {y_labels.shape}")
print(f"Example y_labels[0]: {y_labels[0]}")
print(f"Labels document 0: {string_labels[y_labels[0] == 1]}\n")

print(f"Total documents: {len(all_documents)}")
print("-" * 30)
print(f"Train Set:      {len(train_docs)}")
print(f"Validation Set: {len(val_docs)}")
print(f"Test Set:       {len(test_docs)}")

## Curriculum learning split

We will split the training in 2 phases. Phase 1 -> Only mentioned documents, where every chunk will have the code from the mention annotation. Phase 2-> Not mentioned Documents, where we will use multiple instance learning using maxpool

In [ ]:
train_docs_phase1 = []
y_train_phase1 = []

train_docs_phase2 = []
y_train_phase2 = []

# Only the training set
for doc, y in zip(train_docs, y_train):
    if doc.mentions and len(doc.mentions)>0:
        # Strong supervision
        train_docs_phase1.append(doc)
        y_train_phase1.append(y)
    else:
        # Weak Supervision
        train_docs_phase2.append(doc)
        y_train_phase2.append(y)

y_train_phase1 = np.array(y_train_phase1)
y_train_phase2 = np.array(y_train_phase2)

print("\n=== CURRICULUM LEARNING SPLIT ===")
print(f"Phase 1 Train Docs (With Mentions): {len(train_docs_phase1)}")
print(f"Phase 2 Train Docs (No Mentions):   {len(train_docs_phase2)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('nlpaueb/bert-base-greek-uncased-v1', do_lower_case = True, strip_accents = True)

## Tokenization/ Sliding Window/ BIO Tagging

We will use the greekBERT tokenizer to tokenize the text. In the previous section we saw that a lot of documents have more than 512 tokens. So we will use a sliding window technique to convert the big documents to chunks. After we will tag the tokens with the mentions we have using the BIO tagging technique. The fact that we will create chunks from the documents creates a problem with the labels. There is not a single way to solve this , so we will experiment with this

In [ ]:
def prepare_features(docs, y_labels, tokenizer, string_labels):
    all_input_ids = []
    all_attention_masks = []
    all_ner_labels = []     #  BIO tags
    all_class_labels = []   # multi-hot encoding
    all_patient_ids = []

    for doc_idx, doc in enumerate(docs):
        doc_y = y_labels[doc_idx]
        
        # 2. Tokenization & Sliding Window
        encoded = tokenizer(
            doc.text,
            padding='max_length',
            truncation=True,
            max_length=512,
            stride=64,
            return_overflowing_tokens=True,
            return_offsets_mapping=True
        )
        
        # for every chunk in the document
        for chunk_idx in range(len(encoded['input_ids'])):
            chunk_input_ids = encoded['input_ids'][chunk_idx]
            chunk_mask = encoded['attention_mask'][chunk_idx]
            chunk_offsets = encoded['offset_mapping'][chunk_idx]
            
            
            # Initialize BIO tags
            chunk_ner_tags = ['O'] * 512
            
            # Initialize the labels 
            chunk_class_label = np.zeros(110, dtype=int)
            
            # If the document has mention annotation for the labels
            if doc.mentions and len(doc.mentions) > 0:
                for i, (start, end) in enumerate(chunk_offsets):
                    
                    # Check if special token
                    if start==end==0:
                        chunk_ner_tags[i] = -100
                        continue
                    
                    for mention in doc.mentions:
                        if start >= mention.start and end <= mention.end:
                            if i == 0 or chunk_ner_tags[i-1] not in [f"B-{mention.code}", f"I-{mention.code}"]:
                                chunk_ner_tags[i] = f"B-{mention.code}"
                            else:
                                chunk_ner_tags[i] = f"I-{mention.code}"
                            index = np.where(string_labels == mention.code)
                            chunk_class_label[index] = 1
                            break
                    pass 
            else:
                # If the document does not have mentions then we set all tags to -100 so pytorch will ignore them
                chunk_ner_tags = [-100] * 512
                
                chunk_class_label = doc_y
                pass
                
            # Save the chunks results to the lists
            all_patient_ids.append(doc.patient_id)
            all_input_ids.append(chunk_input_ids)
            all_attention_masks.append(chunk_mask)
            all_ner_labels.append(chunk_ner_tags)
            all_class_labels.append(chunk_class_label)

    return all_input_ids, all_attention_masks, all_ner_labels, all_class_labels, all_patient_ids
    

## Convert the data to tensors

Let's say that this section is where analysis and experiments stop and starts the 'real' nlp task. To summarize our previous work:

1. We have 3 sets. Training/Validation/Test.

2. We have 4 lists: input_ids, attention_masks, ner_labels, class_labels, for each set.

3. We used a sliding window to create tokens_sets of max 512 tokens and to keep all information, we didnt truncate but we splitted the documents to chunks

4. The chunks of documents with mentions will have the labels from the mentions for each chunk

5. The chunks of documents without mentions will inherit the labels from the global document (it is possible that this way add false information).

#### Convert BIO Tags to numbers

Create tag to id dictionary

In [ ]:
tag2id_dict = {}
tag2id_dict['O'] = 0
counter = 0
for code in string_labels:
    counter+=1
    tag2id_dict['B-' + code] = counter
    counter+=1
    tag2id_dict['I-' + code] = counter


In [ ]:
def convert_tag2id(ner_labels,tag2id_dict):
    id_list = []
    for i in range(len(ner_labels)):
        if ner_labels[i] != -100:
            id_list.append(tag2id_dict[ner_labels[i]])
        else:
            id_list.append(-100)
    return id_list

#### Create tensors

In [ ]:
def create_tensors(x_docs,y,tokenizer,string_labels,tag2id_dict):
    input_ids, attention_masks, ner_labels, labels , patients_ids= prepare_features(x_docs,y,tokenizer,string_labels)
    for i in range(len(ner_labels)):
        ner_labels[i] = convert_tag2id(ner_labels[i],tag2id_dict)

    input_ids = np.array(input_ids)
    attention_masks = np.array(attention_masks)
    ner_labels = np.array(ner_labels)
    labels = np.array(labels)
    patients_ids = np.array(patients_ids)
    return torch.tensor(input_ids).long() , torch.tensor(attention_masks).long() , torch.tensor(ner_labels).long() , torch.tensor(labels, dtype=torch.float32) , torch.tensor(patients_ids, dtype=torch.int)

In [ ]:
train_input_ids_tensor, train_attention_masks_tensor, train_ner_labels_tensor, y_train_tensor, patient_ids_train = create_tensors(train_docs,y_train,tokenizer,string_labels,tag2id_dict)

In [ ]:
train1_input_ids_tensor, train1_attention_masks_tensor, train1_ner_labels_tensor, y_train1_tensor, patient_ids_train1 = create_tensors(train_docs_phase1,y_train_phase1,tokenizer,string_labels,tag2id_dict)
val_input_ids_tensor, val_attention_masks_tensor, val_ner_labels_tensor, y_val_tensor, patient_ids_val = create_tensors(val_docs,y_val,tokenizer,string_labels,tag2id_dict)
test_input_ids_tensor, test_attention_masks_tensor, test_ner_labels_tensor, y_test_tensor , patient_ids_test= create_tensors(test_docs,y_test,tokenizer,string_labels,tag2id_dict)

In [ ]:
print(f"Phase 1 Training set shape: {train1_input_ids_tensor.shape}")
print(f"Phase 2 Training set shape: {train_input_ids_tensor.shape}")

In [ ]:
print(f"Input id shape {train1_input_ids_tensor.shape}")
print(f"Attention mask shape {train1_attention_masks_tensor.shape}")
print(f"NER labels shape {train1_ner_labels_tensor.shape}")
print(f"Labels shape {y_train1_tensor.shape}")

#### Create TensorDataset and DataLoader

In [ ]:
BATCH_SIZE = 16


train_dataset1 = TensorDataset(train1_input_ids_tensor, train1_attention_masks_tensor, train1_ner_labels_tensor, y_train1_tensor, patient_ids_train1)
train_loader_phase1 = DataLoader(train_dataset1,batch_size=BATCH_SIZE,shuffle=True)
val_dataset = TensorDataset(val_input_ids_tensor, val_attention_masks_tensor, val_ner_labels_tensor, y_val_tensor,patient_ids_val)
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)
test_dataset = TensorDataset(test_input_ids_tensor, test_attention_masks_tensor, test_ner_labels_tensor, y_test_tensor, patient_ids_test)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)


In [ ]:
train_dataset = TensorDataset(train_input_ids_tensor, train_attention_masks_tensor, train_ner_labels_tensor, y_train_tensor, patient_ids_train)
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)

## Model

We will use the GreekBERT from hugging face

In [ ]:
class ElCardioModel(nn.Module):
    def __init__(self):
        super(ElCardioModel, self).__init__()

        # Greek Bert with no heads
        self.bert = AutoModel.from_pretrained("nlpaueb/bert-base-greek-uncased-v1")

        # Dropout for regularization
        self.dropout = nn.Dropout(p=0.1)

        # Classification Head
        self.classification_head = nn.Linear(768,110)

        # NER Head
        self.ner_head = nn.Linear(768,221)


    def forward(self, input_ids, attention_mask):
        
        bert_output = self.bert(input_ids,attention_mask)
        sequence_bert_output = bert_output.last_hidden_state 

        sequence_bert_output = self.dropout(sequence_bert_output)
        # classification head
        classification = self.classification_head(sequence_bert_output[:, 0, :])

        # NER head
        ner = self.ner_head(sequence_bert_output)

        return classification, ner

## Loss Function

### Weights

In [ ]:
# Positive samples for each class
train_counts = y_train.sum(axis=0)
total_samples = y_train.shape[0]

# Calculate weights
pos_weights = (total_samples - train_counts) / np.clip(train_counts, 1, None)

# CLIPPING: max wieght = 40 so we dont explode the gradients
pos_weights_clipped = np.clip(pos_weights, 1.0, 30.0)

pos_weights_tensor = torch.tensor(pos_weights_clipped, dtype=torch.float32).to(device)

print("Examples of weights")
print(np.round(pos_weights_clipped[:30], 2))

#### Focal Loss

In [ ]:
def focal_binary_cross_entropy(logits, targets, gamma=2.0, pos_weight = None):
    # Calculate the BCE Loss for each label and for each document separate

    bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none', pos_weight = pos_weight)
    
    # Calculate the probabilities
    p = torch.sigmoid(logits)
    
    p_t = torch.where(targets >= 0.5, p, 1.0 - p)
    
    # modulating factor
    focal_weight = (1.0 - p_t) ** gamma
    
    final_loss = (focal_weight * bce_loss).mean()
    
    num_labels = logits.shape[-1]
    return final_loss * num_labels

#### Cross Entropy for ner

In [ ]:
def ner_cross_entropy(logits,targets):

    logits_2d = logits.view(-1,logits.size(-1))
    targets_1d = targets.view(-1)

    return F.cross_entropy(logits_2d,targets_1d,ignore_index=-100)


#### Total Loss

In [ ]:
def calculate_loss(logits,targets):

    a = 1
    b = 1

    return (a*focal_binary_cross_entropy(logits,targets)) + (b*ner_cross_entropy(logits,targets))

## Optimizer

In [ ]:
model = ElCardioModel()
parameters = []
bert_params = {'params': model.bert.parameters() , 'lr' : 1e-5, 'weight_decay' : 0.01}
classification_head_params = {'params': model.classification_head.parameters() , 'lr' : 1e-3, 'weight_decay': 0.1}
ner_head_params = {'params': model.ner_head.parameters() , 'lr' : 1e-3, 'weight_decay': 0.05}
parameters.append(bert_params)
parameters.append(classification_head_params)
parameters.append(ner_head_params)

optimizer = AdamW(parameters)

## Mentioned Docs Training Loop

In [ ]:
# --- Configuration & Hyperparameters ---
epochs = 10
alpha = 0.5 # Weight for Classification Loss
beta = 20 # Weight for NER Loss
best_val_loss = float('inf') # Initialize with infinity

# Lists to store the loss values for plotting later
train_loss = []
val_loss = []

history_val_loss_c = []
history_val_loss_n = []

model.to(device)

print(f"Starting training on {device} for {epochs} epochs...\n")

# --- Main Epoch Loop ---
for epoch in range(epochs):
    print(f"======== Epoch {epoch + 1} / {epochs} ========")
    
    model.train() # Set model to training mode (enables Dropout)
    total_train_loss = 0
    
    # Initialize progress bar for the training loader
    train_progress = tqdm(train_loader_phase1, desc="Training")
    
    for batch in train_progress:
        # 1. Unpack the batch and move tensors to the device
        b_input_ids = batch[0].to(device)
        b_attn_mask = batch[1].to(device)
        b_ner_labels = batch[2].to(device)
        b_class_labels = batch[3].to(device)
        
        # 2. Clear previously calculated gradients
        optimizer.zero_grad()
        
        # 3. Forward pass: compute predictions
        logits_class, logits_ner = model(b_input_ids, b_attn_mask)
        
        # 4. Calculate the two separate losses
        loss_c = alpha*focal_binary_cross_entropy(logits_class, b_class_labels,pos_weight = pos_weights_tensor)
        loss_n = beta*ner_cross_entropy(logits_ner, b_ner_labels)
        
        # 5. Combine the losses using the predefined weights
        total_loss =  loss_c + loss_n
        
        # 6. Backward pass: compute gradients
        total_loss.backward()
        
        # 7. Update weights
        optimizer.step()
        
        # Accumulate the training loss
        total_train_loss += total_loss.item()
        
        # Update progress bar with the current batch loss
        train_progress.set_postfix({'batch_loss': total_loss.item()})
        
    # Calculate average training loss for the epoch
    avg_train_loss = total_train_loss / len(train_loader_phase1)
    train_loss.append(avg_train_loss)
    
    
    # VALIDATION

    model.eval() # Set model to evaluation mode (disables Dropout)
    total_val_loss_n = 0
    valid_ner_batches = 0

    # Lists to save the data from the epoch
    all_val_logits = []
    all_val_targets = []
    all_val_patient_ids = []

    # Initialize progress bar for the validation loader
    val_progress = tqdm(val_loader, desc="Validation")
    
    with torch.no_grad():
        for batch in val_progress:
            # 1. Unpack and move to device
            b_input_ids = batch[0].to(device)
            b_attn_mask = batch[1].to(device)
            b_ner_labels = batch[2].to(device)
            b_class_labels = batch[3].to(device)
            b_patient_ids = batch[4]
            
            # 2. Forward pass
            logits_class, logits_ner = model(b_input_ids, b_attn_mask)
            
            # 3. Calculate ner losses
            loss_n = ner_cross_entropy(logits_ner, b_ner_labels)
            if not torch.isnan(loss_n):
                total_val_loss_n += loss_n.item()
                valid_ner_batches +=1
            
            # 4. Classification
            all_val_logits.append(logits_class.cpu())
            all_val_targets.append(b_class_labels.cpu())
            all_val_patient_ids.append(b_patient_ids.cpu())

    # Concat the lists to big pytorch tensors
    cat_logits = torch.cat(all_val_logits).numpy()
    cat_targets = torch.cat(all_val_targets).numpy()
    cat_ids = torch.cat(all_val_patient_ids).numpy()  

    df_logits = pd.DataFrame(cat_logits)
    df_logits['patient_id'] = cat_ids
    
    df_targets = pd.DataFrame(cat_targets)
    df_targets['patient_id'] = cat_ids

    # Maxpool per document
    doc_logits_df = df_logits.groupby('patient_id').max()
    doc_targets_df = df_targets.groupby('patient_id').max()

    doc_logits_tensor = torch.tensor(doc_logits_df.values).to(device)
    doc_targets_tensor = torch.tensor(doc_targets_df.values).to(device)

    # Classification loss
    doc_loss_c = focal_binary_cross_entropy(doc_logits_tensor, doc_targets_tensor)

    if valid_ner_batches > 0:
        # Average Ner loss (per chunks)
        avg_val_loss_n = total_val_loss_n / valid_ner_batches
    else:
        avg_val_loss_n = 0.0

    # Final loss
    avg_val_loss = (alpha * doc_loss_c.item()) + (beta * avg_val_loss_n)

    val_loss.append(avg_val_loss)
    history_val_loss_c.append(alpha * doc_loss_c.item())
    history_val_loss_n.append(beta * avg_val_loss_n)
    print(f"\nAverage Training Loss: {avg_train_loss:.4f}")
    print(f"Average Validation Loss: {avg_val_loss:.4f}")

    print(f"classifcation losses {history_val_loss_c}")
    print(f"ner losses {history_val_loss_n}")

    # Save the model only if the validation loss improved
    if avg_val_loss < best_val_loss:
        print(f"Validation loss improved from {best_val_loss:.4f} to {avg_val_loss:.4f}. Saving model...")
        best_val_loss = avg_val_loss
        # Save the current state of the model's weights
        torch.save(model.state_dict(), 'best_elcardio_model_phase.pth')
    else:
        print(f"Validation loss did not improve.")
    print("-" * 50 + "\n")

print("Training Complete! The best model weights are saved in 'best_elcardio_model.pth'.")

In [ ]:
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(14, 5))

# --- Plot 1: Train vs Val ---
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_loss, label='Train Total Loss', marker='o', color='blue')
plt.plot(epochs_range, val_loss, label='Val Total Loss', marker='o', color='orange')
plt.title('Overfitting Check')
plt.xlabel('Epochs')
plt.ylabel('Total Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# --- Plot 2: Compare scale Classification vs NER ---
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history_val_loss_c, label='Classification Loss (Focal)', marker='s', color='green')
plt.plot(epochs_range, history_val_loss_n, label='NER Loss (CrossEntropy)', marker='^', color='red')
plt.title(f"Sanity Check with a={alpha} b={beta}")
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:


epochs_final = 5
alpha = 2
beta = 20

model.load_state_dict(torch.load('best_elcardio_model_phase.pth'))

parameters_phase2 = []
bert_params_p2 = {'params': model.bert.parameters() , 'lr': 1e-6, 'weight_decay': 0.01}
class_head_params_p2 = {'params': model.classification_head.parameters() , 'lr': 1e-3, 'weight_decay': 0.005}
ner_head_params_p2 = {'params': model.ner_head.parameters() , 'lr': 1e-4, 'weight_decay': 0.1}

parameters_phase2.append(bert_params_p2)
parameters_phase2.append(class_head_params_p2)
parameters_phase2.append(ner_head_params_p2)

optimizer = AdamW(parameters_phase2)

best_val_loss = float('inf')

model.to(device)

print(f"Starting Final Fine-Tuning on {device} for {epochs_final} epochs...\n")

# --- Main Epoch Loop ---
for epoch in range(epochs_final):
    print(f"======== Epoch {epoch + 1} / {epochs_final} ========")
    
    model.train() 
    total_train_loss = 0
    
    # Original train_loader
    train_progress = tqdm(train_loader, desc="Training (Rehearsal)")
    
    for batch in train_progress:
        b_input_ids = batch[0].to(device)      
        b_attn_mask = batch[1].to(device)      
        b_ner_labels = batch[2].to(device)     
        b_class_labels = batch[3].to(device)   
        
        # 2. Clear gradients
        optimizer.zero_grad()
        
        # 3. Forward pass
        logits_class, logits_ner = model(b_input_ids, b_attn_mask)
 
        # 4. Losses
        loss_c = alpha * focal_binary_cross_entropy(logits_class, b_class_labels,gamma=2.5,pos_weight = pos_weights_tensor)
        loss_n = ner_cross_entropy(logits_ner, b_ner_labels)
        
        total_loss = loss_c
        if not torch.isnan(loss_n):
            total_loss += beta * loss_n
        
        # 5. Backward pass
        total_loss.backward()
        
        # 6. Update weights
        optimizer.step()
        
        total_train_loss += total_loss.item()
        train_progress.set_postfix({'batch_loss': total_loss.item()})
        
    avg_train_loss = total_train_loss / len(train_loader)
    train_loss.append(avg_train_loss)
    
    # ==========================================
    # VALIDATION
    # ==========================================
    model.eval() 
    total_val_loss_n = 0
    valid_ner_batches = 0

    all_val_logits = []
    all_val_targets = []
    all_val_patient_ids = []

    val_progress = tqdm(val_loader, desc="Validation")
    
    with torch.no_grad():
        for batch in val_progress:
            b_input_ids = batch[0].to(device)
            b_attn_mask = batch[1].to(device)
            b_ner_labels = batch[2].to(device)
            b_class_labels = batch[3].to(device)
            b_patient_ids = batch[4]
            
            logits_class, logits_ner = model(b_input_ids, b_attn_mask)
            
            # NER loss calc
            loss_n = ner_cross_entropy(logits_ner, b_ner_labels)
            if not torch.isnan(loss_n):
                total_val_loss_n += loss_n.item()
                valid_ner_batches += 1
            
            # Classification
            all_val_logits.append(logits_class.cpu())
            all_val_targets.append(b_class_labels.cpu())
            all_val_patient_ids.append(b_patient_ids.cpu())

    cat_logits = torch.cat(all_val_logits).numpy()
    cat_targets = torch.cat(all_val_targets).numpy()
    cat_ids = torch.cat(all_val_patient_ids).numpy()  

    df_logits = pd.DataFrame(cat_logits)
    df_logits['patient_id'] = cat_ids
    
    df_targets = pd.DataFrame(cat_targets)
    df_targets['patient_id'] = cat_ids

    doc_logits_df = df_logits.groupby('patient_id').max()
    doc_targets_df = df_targets.groupby('patient_id').max()

    doc_logits_tensor = torch.tensor(doc_logits_df.values).to(device)
    doc_targets_tensor = torch.tensor(doc_targets_df.values).to(device)

    doc_loss_c = focal_binary_cross_entropy(doc_logits_tensor, doc_targets_tensor)

    if valid_ner_batches > 0:
        avg_val_loss_n = total_val_loss_n / valid_ner_batches
    else:
        avg_val_loss_n = 0.0

    avg_val_loss = (alpha * doc_loss_c.item()) + (beta * avg_val_loss_n)

    val_loss.append(avg_val_loss)
    history_val_loss_c.append(alpha * doc_loss_c.item())
    history_val_loss_n.append(beta * avg_val_loss_n)
    
    print(f"\nAverage Training Loss: {avg_train_loss:.4f}")
    print(f"Average Validation Loss: {avg_val_loss:.4f}")
    print(f"Classification losses {history_val_loss_c}")
    print(f"NER losses {history_val_loss_n}")

    if avg_val_loss < best_val_loss:
        print(f"Validation loss improved from {best_val_loss:.4f} to {avg_val_loss:.4f}. Saving model...")
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_elcardio_model_final.pth')
    else:
        print(f"Validation loss did not improve.")
    print("-" * 50 + "\n")

print("Training Complete! The best model weights are saved in 'best_elcardio_model_final.pth'.")

## Testing

In [ ]:
# --- 1. Load the Best Model ---
# Load the weights that were saved during the best validation epoch
model.load_state_dict(torch.load('best_elcardio_model_final.pth'))
model.eval() # Set to evaluation mode (critical!)

# Lists to store the raw predictions and true labels
all_probs = []
all_targets = []
all_patient_ids = []

print("Starting inference on the Test Set...")

# Disable gradient calculation for testing
with torch.no_grad():
    for batch in test_loader: 
        # Unpack and move to device
        b_input_ids = batch[0].to(device)
        b_attn_mask = batch[1].to(device)
        b_class_labels = batch[3].to(device) # Index 3 is classification targets
        b_patient_ids = batch[4]
        
        # Forward pass
        logits_class, _ = model(b_input_ids, b_attn_mask)
        
        # Apply Sigmoid to convert logits to probabilities (0 to 1)
        probs = torch.sigmoid(logits_class)
        
        # Convert probabilities to binary predictions using a 0.5 threshold
        #preds = (probs > 0.5).int()
        
        # Move back to CPU and convert to numpy arrays for sklearn
        all_probs.append(probs.cpu().numpy())
        all_targets.append(b_class_labels.cpu().numpy())
        all_patient_ids.append(b_patient_ids.numpy())

# Stack the lists into large 2D numpy arrays: shape (num_samples, 110_classes)
all_probs = np.vstack(all_probs)
all_targets = np.vstack(all_targets)
all_patient_ids = np.concatenate(all_patient_ids)


# Create Dataframes for the predictions using patient_id 
df_probs = pd.DataFrame(all_probs)
df_probs['patient_id'] = all_patient_ids

# Create Dataframes for the test labels using patient_id 
df_targets = pd.DataFrame(all_targets)
df_targets['patient_id'] = all_patient_ids

# Max pooling per document
doc_probs_df = df_probs.groupby('patient_id').max()
doc_targets_df = df_targets.groupby('patient_id').max()

# Convert to numpy arrays without the patient id column
final_document_probs = doc_probs_df.values
final_document_targets = doc_targets_df.values




In [ ]:
final_document_preds = (final_document_probs > 0.3).astype(int)

# --- 2. Calculate Global Metrics (Micro & Macro) ---
print("\n" + "="*50)
print("GLOBAL TEST METRICS (CLASSIFICATION)")
print("="*50)

# Calculate Micro metrics
micro_p, micro_r, micro_f1, _ = precision_recall_fscore_support(final_document_targets, final_document_preds, average='micro', zero_division=0)
print(f"MICRO - Precision: {micro_p:.4f} | Recall: {micro_r:.4f} | F1-Score: {micro_f1:.4f}")

# Calculate Macro metrics
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(final_document_targets, final_document_preds, average='macro', zero_division=0)
print(f"MACRO - Precision: {macro_p:.4f} | Recall: {macro_r:.4f} | F1-Score: {macro_f1:.4f}")

In [ ]:

# Find classes in the test set
class_frequencies = final_document_targets.sum(axis=0)
valid_class_indices = np.where(class_frequencies > 0)[0]
total_true_preds = 0
class_stats = []

for i in valid_class_indices:
    # Real classes and preds for class i
    class_name = string_labels[i]
    true_y = final_document_targets[:, i]
    pred_y = final_document_preds[:, i]
    probs_y = final_document_probs[:, i]
    total_true_preds += len(pred_y[pred_y==1])
    # Υπολογισμός TP, FP, FN
    TP = np.sum((true_y == 1) & (pred_y == 1))
    FP = np.sum((true_y == 0) & (pred_y == 1))
    FN = np.sum((true_y == 1) & (pred_y == 0))
    Support = np.sum(true_y == 1)
    
    # MOdels confidence
    positive_probs = probs_y[true_y == 1]
    avg_confidence_when_true = np.mean(positive_probs) if len(positive_probs) > 0 else 0.0
    
    # F1 Score
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    class_stats.append({
        'Code': class_name,
        'Support': Support,
        'True Positives': TP,
        'False Negatives (Missed)': FN,
        'False Positives (Wrongly Guessed)': FP,
        'Precision': round(precision, 3),
        'Recall': round(recall, 3),
        'F1_Score': round(f1, 3),
        'Avg_Confidence_When_True': round(avg_confidence_when_true, 3)
    })

df_error_analysis = pd.DataFrame(class_stats)

# less frequent classes
print("=== ERROR ANALYSIS: 20 RAREST CLASSES ===")
display(df_error_analysis.sort_values(by=['Support', 'Code']).head(20))

# most frequent classes
print("\n=== ERROR ANALYSIS: 10 MOST FREQUENT CLASSES ===")
display(df_error_analysis.sort_values(by='Support', ascending=False).head(10))

In [ ]:
print(total_true_preds)